# Comprobacion HSI

Comprobacion visual de la segmentacion HSI para `SB012`, `SB013`, `SB017`, `SB018`, `SB019` y `SB020`.

Carga las funciones de `9_funcion_preprocesado.ipynb`, igual que `10_guardar_imagenes_a_escala.ipynb`, y muestra pseudo-RGB original vs HSI segmentada lado a lado.

In [ ]:
from pathlib import Path
import warnings

import cv2
import matplotlib.pyplot as plt
import nbformat
import numpy as np
from PIL import Image
from spectral import open_image

plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path(r'Registration\DeepLearning')
PREPROCESS_NOTEBOOK = BASE_DIR / '9_funcion_preprocesado.ipynb'
OUTPUT_DIR = BASE_DIR / 'Comprobaciones_segmentacion' / 'HSI'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Notebook de funciones:', PREPROCESS_NOTEBOOK)
print('Carpeta de salida:', OUTPUT_DIR)

## Cargar funciones del notebook 9

In [ ]:
def load_preprocess_definitions(notebook_path):
    nb = nbformat.read(str(notebook_path), as_version=4)
    skipped = []

    for idx, cell in enumerate(nb.cells):
        if cell.cell_type != 'code':
            continue

        source = cell.source
        skip_patterns = [
            'preprocess_output = preprocesar_he_hsi',
            'validation_results = {}',
        ]
        if any(pattern in source for pattern in skip_patterns):
            skipped.append(idx)
            continue

        exec(compile(source, f'{notebook_path.name}:cell_{idx}', 'exec'), globals())

    print(f'Celdas de ejemplo/validacion saltadas: {skipped}')


load_preprocess_definitions(PREPROCESS_NOTEBOOK)

## Funciones de pseudo-RGB y visualizacion

In [ ]:
def robust_normalize_for_display(channel, p_low=2, p_high=98, gamma=1.0):
    ch = np.asarray(channel, dtype=np.float32)
    ch = np.where(np.isfinite(ch), ch, np.nan)
    if np.all(np.isnan(ch)):
        return np.zeros_like(ch, dtype=np.float32)
    lo = np.nanpercentile(ch, p_low)
    hi = np.nanpercentile(ch, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(ch, dtype=np.float32)
    ch = np.clip((ch - lo) / (hi - lo), 0, 1)
    if gamma != 1.0:
        ch = ch ** gamma
    return np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def load_hsi_cube_and_wavelengths(hdr_path):
    img = open_image(str(hdr_path))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array([float(w) for w in img.metadata['wavelength']], dtype=np.float32)
    return cube, wavelengths


def load_hsi_pseudo_rgb(hdr_path, r_nm=650, g_nm=550, b_nm=450):
    cube, wavelengths = load_hsi_cube_and_wavelengths(hdr_path)

    def nearest_band(target_nm):
        return int(np.argmin(np.abs(wavelengths - target_nm)))

    r_idx = nearest_band(r_nm)
    g_idx = nearest_band(g_nm)
    b_idx = nearest_band(b_nm)

    rgb = np.stack([
        robust_normalize_for_display(cube[:, :, r_idx]),
        robust_normalize_for_display(cube[:, :, g_idx]),
        robust_normalize_for_display(cube[:, :, b_idx]),
    ], axis=-1)
    rgb_u8 = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

    info = {
        'display_mode': 'pseudo_rgb',
        'shape': rgb_u8.shape,
        'r_idx': r_idx,
        'g_idx': g_idx,
        'b_idx': b_idx,
        'r_nm': float(wavelengths[r_idx]),
        'g_nm': float(wavelengths[g_idx]),
        'b_nm': float(wavelengths[b_idx]),
    }
    return rgb_u8, info


def load_hsi_grayscale(hdr_path, min_nm=800, max_nm=950, p_low=0.2, p_high=99.8, gamma=0.75):
    cube, wavelengths = load_hsi_cube_and_wavelengths(hdr_path)
    band_idx = np.where((wavelengths >= min_nm) & (wavelengths <= max_nm))[0]
    if band_idx.size == 0:
        band_idx = np.arange(cube.shape[2])

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        gray_float = np.nanmean(cube[:, :, band_idx], axis=2)

    gray_float = np.nan_to_num(gray_float, nan=0.0, posinf=0.0, neginf=0.0)
    gray_u8 = (robust_normalize_for_display(
        gray_float,
        p_low=p_low,
        p_high=p_high,
        gamma=gamma,
    ) * 255).astype(np.uint8)
    if min_nm <= 436 and max_nm >= 1700:
        display_mode = 'hyperlab_full_range'
    elif min_nm >= 750:
        display_mode = 'nir_grayscale'
    else:
        display_mode = 'wide_grayscale'

    info = {
        'display_mode': display_mode,
        'shape': gray_u8.shape,
        'min_nm': float(wavelengths[band_idx[0]]),
        'max_nm': float(wavelengths[band_idx[-1]]),
        'num_bands': int(band_idx.size),
        'p_low': float(p_low),
        'p_high': float(p_high),
        'gamma': float(gamma),
    }
    return gray_u8, info


def gray_to_rgb(gray_u8):
    return np.repeat(np.asarray(gray_u8, dtype=np.uint8)[:, :, None], 3, axis=2)


def largest_component(mask):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num_labels <= 1:
        return mask_u8.astype(bool)

    h, w = mask_u8.shape
    center = np.array([w / 2, h / 2], dtype=np.float32)
    best_label = 1
    best_score = -np.inf
    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area < 1500:
            continue
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        bw = stats[label, cv2.CC_STAT_WIDTH]
        bh = stats[label, cv2.CC_STAT_HEIGHT]
        distance = np.linalg.norm(centroids[label] - center) / max(h, w)
        is_strip = (bw > 0.92 * w and bh < 0.25 * h) or (bh > 0.92 * h and bw < 0.25 * w)
        score = area * (1 - 0.2 * distance) * (0.3 if is_strip else 1.0)
        if score > best_score:
            best_score = score
            best_label = label

    return labels == best_label


GRAYSCALE_HSI_SUBJECTS = {'SB017', 'SB018', 'SB019', 'SB020'}
REFERENCE_HSI_SIZE = (1024, 1280)  # alto, ancho de estas HSI
WIDE_MASK_TRAY_CONFIGS = {
    # La mascara se calcula sobre la misma gris HyperLAB-like que se muestra.
    # La ROI limita la bandeja para evitar que el marco/plastico entre en el contorno.
    'SB017': {'roi': (460, 260, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB018': {'roi': (460, 265, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB019': {'roi': (450, 275, 1010, 790), 'border': 10, 'threshold_factor': 0.55},
    'SB020': {'roi': (455, 280, 1000, 790), 'border': 10, 'threshold_factor': 0.55},
}


def scale_roi_to_image(roi, image_shape):
    ref_h, ref_w = REFERENCE_HSI_SIZE
    h, w = image_shape[:2]
    x1, y1, x2, y2 = roi
    scaled = (
        int(round(x1 * w / ref_w)),
        int(round(y1 * h / ref_h)),
        int(round(x2 * w / ref_w)),
        int(round(y2 * h / ref_h)),
    )
    sx1, sy1, sx2, sy2 = scaled
    sx1 = max(0, min(w - 2, sx1))
    sx2 = max(sx1 + 2, min(w, sx2))
    sy1 = max(0, min(h - 2, sy1))
    sy2 = max(sy1 + 2, min(h, sy2))
    return sx1, sy1, sx2, sy2


def segment_hsi_grayscale_tray(gray_u8, subject_id):
    if subject_id not in WIDE_MASK_TRAY_CONFIGS:
        raise ValueError(f'No hay configuracion grayscale definida para {subject_id}')

    config = WIDE_MASK_TRAY_CONFIGS[subject_id]
    x1, y1, x2, y2 = scale_roi_to_image(config['roi'], gray_u8.shape)
    roi = gray_u8[y1:y2, x1:x2]
    valid_values = roi[(roi > 5) & (roi < 250)]
    if valid_values.size > 100:
        otsu_value, _ = cv2.threshold(
            valid_values.reshape(-1, 1).astype(np.uint8),
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU,
        )
    else:
        otsu_value = 80.0

    threshold_factor = float(config.get('threshold_factor', 0.70))
    threshold = max(34.0, min(95.0, float(otsu_value) * threshold_factor))
    roi_mask = (roi > threshold) & (roi < 245)

    border = max(6, int(round(config.get('border', 10) * min(roi.shape[:2]) / 520)))
    roi_mask[:border, :] = False
    roi_mask[-border:, :] = False
    roi_mask[:, :border] = False
    roi_mask[:, -border:] = False

    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    close_kernel_1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    close_kernel_2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_OPEN, open_kernel) > 0
    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_kernel_1) > 0
    roi_mask = largest_component(roi_mask)
    roi_mask = cv2.morphologyEx(roi_mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, close_kernel_2) > 0
    roi_mask = cv2.dilate(roi_mask.astype(np.uint8), dilate_kernel, iterations=1) > 0

    final_border = max(2, border // 6)
    roi_mask[:final_border, :] = False
    roi_mask[-final_border:, :] = False
    roi_mask[:, :final_border] = False
    roi_mask[:, -final_border:] = False

    mask = np.zeros(gray_u8.shape, dtype=bool)
    mask[y1:y2, x1:x2] = roi_mask

    segmented_rgb = gray_to_rgb(gray_u8)
    segmented_rgb[~mask] = 0

    info = {
        'selected_method': 'hyperlab_gray_roi_mask',
        'selected_because': 'same_hyperlab_gray_display_used_for_contour',
        'roi_xyxy': (int(x1), int(y1), int(x2), int(y2)),
        'otsu_value': float(otsu_value),
        'threshold': float(threshold),
        'threshold_factor': threshold_factor,
        'mask_area_px': int(np.count_nonzero(mask)),
    }
    return segmented_rgb, mask, info


def segment_hsi_for_check(subject_id, hsi_hdr_path):
    if subject_id in GRAYSCALE_HSI_SUBJECTS:
        display_gray, original_info = load_hsi_grayscale(
            hsi_hdr_path,
            min_nm=435,
            max_nm=1700,
            p_low=0.1,
            p_high=99.9,
            gamma=0.70,
        )
        original_rgb = gray_to_rgb(display_gray)
        _, mask, seg_info = segment_hsi_grayscale_tray(display_gray, subject_id)
        segmented_rgb = gray_to_rgb(display_gray)
        segmented_rgb[~mask] = 0
        seg_info.update({
            'display_mode': original_info['display_mode'],
            'grayscale_info': original_info,
            'mask_grayscale_info': original_info,
        })
        return original_rgb, segmented_rgb, mask, original_info, seg_info

    original_rgb, original_info = load_hsi_pseudo_rgb(hsi_hdr_path)
    segmented_rgb, mask, seg_info = extract_specimen_only_from_hsi_path(
        str(hsi_hdr_path),
        grow_pixels=5,
        show_original=False,
        show_result=False,
        return_mask=True,
        return_info=True,
    )
    seg_info = dict(seg_info)
    seg_info['display_mode'] = 'pseudo_rgb'
    return original_rgb, segmented_rgb, mask, original_info, seg_info


def draw_mask_contour(rgb, mask, color=(0, 255, 255), thickness=3):
    out = np.asarray(rgb, dtype=np.uint8).copy()
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, thickness)
    return out


def save_side_by_side(path, original_rgb, segmented_rgb, mask, title, subtitle='', original_label='pseudo-RGB original'):
    segmented_with_contour = draw_mask_contour(segmented_rgb, mask)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), constrained_layout=True)
    axes[0].imshow(original_rgb)
    axes[0].set_title(f'{title} - {original_label}')
    axes[0].axis('off')
    axes[1].imshow(segmented_with_contour)
    axes[1].set_title(f'{title} - segmentada{subtitle}')
    axes[1].axis('off')
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## Comprobacion por sujeto

In [ ]:
hsi_results = {}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    hsi_hdr_path = SPECIMENS[subject_id]['hsi_hdr_path']

    original_rgb, segmented_rgb, mask, original_info, seg_info = segment_hsi_for_check(subject_id, hsi_hdr_path)

    selected_method = seg_info.get('selected_method')
    selected_because = seg_info.get('selected_because')
    display_mode = seg_info.get('display_mode', original_info.get('display_mode', 'pseudo_rgb'))
    if display_mode == 'hyperlab_full_range':
        original_label = 'HyperLAB-like grayscale original'
    elif display_mode == 'nir_grayscale':
        original_label = 'NIR grayscale original'
    elif display_mode in {'grayscale', 'wide_grayscale'}:
        original_label = 'grayscale original'
    else:
        original_label = 'pseudo-RGB original'
    subtitle = f' | {selected_method}' if selected_method else ''

    out_path = OUTPUT_DIR / f'{subject_id}_hsi_original_vs_segmentada.png'
    save_side_by_side(
        out_path,
        original_rgb,
        segmented_rgb,
        mask,
        f'{subject_id} HSI',
        subtitle=subtitle,
        original_label=original_label,
    )

    hsi_results[subject_id] = {
        'hsi_hdr_path': hsi_hdr_path,
        'original_shape': original_rgb.shape,
        'segmented_shape': segmented_rgb.shape,
        'mask_area_px': int(np.count_nonzero(mask)),
        'display_mode': display_mode,
        'original_info': original_info,
        'segmentation_info': seg_info,
        'selected_method': selected_method,
        'selected_because': selected_because,
        'plot_path': out_path,
    }

    print('Modo visualizacion:', display_mode)
    print('Original:', original_rgb.shape)
    print('Segmentada:', segmented_rgb.shape, '| area mascara:', int(np.count_nonzero(mask)))
    print('Metodo:', selected_method, '| motivo:', selected_because)
    if 'roi_xyxy' in seg_info:
        print('ROI:', seg_info['roi_xyxy'], '| threshold:', round(seg_info['threshold'], 2))
    print('Plot:', out_path)

hsi_results
